# Session 8 — Building AI-based Evaluators: LLM-as-a-Judge

**Who checks the checker?**

Session 7 built four agents and measured whether they coordinated. It never asked whether
the diagnosis was any *good*. Today we build the thing that can ask — and then we point it
at itself.

> **The sentence this session adds:** *You cannot measure a difference smaller than your
> judge's own noise.*

Halvard Works is fictional — the plant, its machines, sensors, manuals, parts and history
were invented for this course. Inspired by publicly described industrial-copilot products;
not affiliated with or endorsed by any vendor.

## Every term, before it is used

| term | in this session it means |
|---|---|
| **judge** | an evaluator whose verdict comes from a language model instead of from code. |
| **rubric** | the prompt that tells the judge where to look and what would count as failing. |
| **verdict** | one of exactly three words: `SOUND`, `UNSOUND`, `INSUFFICIENT-EVIDENCE`. |
| **score** | those three words as `1`, `0`, `None`. `None` means *skipped*, as everywhere else. |
| **insufficient-evidence** | the judge's third verdict: *the report does not contain enough for me to rule either way*. It is a real answer, not a failure — and a judge that never returns it is a judge that is guessing. |
| **a report** | what the four agents wrote back to ONE engineer's question: four sections of prose, ending in five structured lines (`MACHINE:`, `FAULT_CODE:`, `SECTION:`, `PART:`, `ACTION:`). Those five lines are all a code check can read. |
| **an arm** | one of seven versions of the same report: five broken on purpose, two not. |
| **separation** | how much more often a judge fires on its own broken arm than on the arms it should pass. |
| **a flip** | the judge changing its answer on the same, *unchanged* report. |
| **flip rate** | how often that happens. Same thing as *wobble*, below — one is the number, one is the name. |
| **the rule of three** | run something `n` times, see **zero** events, and the most you can honestly say is that the rate is below `3/n` (95% upper bound). `n=5` → below 60%; `n=10` → below 30%. Session 5 taught this; today it is applied to the judge itself. |
| **wobble** | how often the same judge, on the same unchanged report, disagrees with its own most common answer. Measured as a flip rate, reported as a rule-of-three bound. |
| **the gate** | separation's lower bound must clear wobble's upper bound. Otherwise the judge is decoration. |
| **a verdict fires** | it returned `0`. Confusingly, that is the judge working. |

In [ ]:
# [PULL]
!git pull --ff-only
!python check_env.py

In [ ]:
# [PINS]
# A resolution FAILURE is loud. A resolution SUCCESS that quietly installed a
# different version is silent, and would corrupt every number below.
#
# THIS CELL IS PROVIDER-AWARE ON PURPOSE. You are on one of three providers and
# the package YOU need is not the package your neighbour needs. Last term a
# student on Gemini got a confusing failure four cells later because nothing had
# checked langchain-google-genai was importable.
import importlib.metadata as md
import os

PROVIDER = os.environ.get('COURSE_PROVIDER', 'anthropic')
PINS = {'langchain-core': '1.6.1', 'langgraph': '1.2.11', 'langsmith': '0.11.1'}
PROVIDER_PIN = {'anthropic': ('langchain-anthropic', '1.6.1'),
                'openai':    ('langchain-openai', '1.6.0'),
                'google':    ('langchain-google-genai', '4.3.7')}
if PROVIDER not in PROVIDER_PIN:
    print(f'COURSE_PROVIDER={PROVIDER!r} is not one of {list(PROVIDER_PIN)} '
          '-- fix .env before going on')
else:
    pkg, want = PROVIDER_PIN[PROVIDER]
    PINS[pkg] = want
    bad = {}
    for name, w in PINS.items():
        try:
            got = md.version(name)
        except md.PackageNotFoundError:
            got = 'NOT INSTALLED'
        if got != w:
            bad[name] = got
    print(f'provider: {PROVIDER}  (needs {pkg}=={want})')
    print('pins OK' if not bad else f'WRONG/MISSING: {bad} -- fix before going on')

## Running on OpenAI or Gemini

Everything in this notebook works on all three providers. You change **one file**,
`.env` at the **repo root** (not in this folder), and then **restart the kernel** —
`.env` is read once at import, so an edit without a restart does nothing and looks
like the edit did not work.

| you want | put this in `.env` | key line | package (already in requirements.txt) |
|---|---|---|---|
| Anthropic | `COURSE_PROVIDER=anthropic` | `ANTHROPIC_API_KEY=sk-...` | `langchain-anthropic` |
| OpenAI | `COURSE_PROVIDER=openai` | `OPENAI_API_KEY=sk-...` | `langchain-openai` |
| Gemini | `COURSE_PROVIDER=google` | `GOOGLE_API_KEY=...` | `langchain-google-genai` |

Then: **Kernel → Restart**, and run from the top. The `[PROVIDER]` cell below prints
exactly what it is using, so you never have to guess whether the change took.

> **Your numbers will not match the slides, and that is the finding, not a bug.**
> Every measured number here came from `claude-sonnet-5`. A different model is a
> different judge: it separates by a different amount and wobbles by a different
> amount. What should survive a change of provider is the **method**, and usually the
> **ordering** — which judges clear their own noise and which do not. The exact
> percentages will not survive. Report yours.

In [ ]:
# [PROVIDER]
# WHOSE NUMBERS ARE ON THE SLIDES, AND WHY YOURS WILL DIFFER.
#
# Every measured number in this notebook and on the slides was produced with
# claude-sonnet-5. If you are on OpenAI or Gemini, your judge is a different
# judge: it will separate by a different amount and wobble by a different amount.
#
# That is not a bug and it is not you doing it wrong. It is the finding. What
# should survive a change of provider is the METHOD and usually the ORDERING --
# which judges clear their own noise and which do not. The exact percentages will
# not survive, and anyone who tells you otherwise has not checked.
import os

import _path  # noqa: F401  -- this cell runs before [SETUP], so bootstrap here too
import evalkit

KEY_OF = {'anthropic': 'ANTHROPIC_API_KEY', 'openai': 'OPENAI_API_KEY',
          'google': 'GOOGLE_API_KEY'}
prov = evalkit.PROVIDER
key = KEY_OF.get(prov, '?')
print('provider  :', prov)
print('model     :', evalkit.MODEL_IDS.get(prov, '?'))
print('key       :', 'loaded' if os.environ.get(key) else f'MISSING ({key})')
print('.env file :', _path.ROOT / '.env',
      '' if (_path.ROOT / '.env').exists() else '   <-- DOES NOT EXIST')
print('slides    : claude-sonnet-5')
print()

# TO SWITCH PROVIDER: edit the two lines below into .env at the REPO ROOT,
# then Kernel -> Restart, then run from the top. Nothing else changes.
if prov != 'anthropic' or not os.environ.get(key):
    print('To switch provider, put these TWO lines in', _path.ROOT / '.env', ':')
    print()
    for p, k in KEY_OF.items():
        mark = '  <-- you are here' if p == prov else ''
        print(f'    COURSE_PROVIDER={p}')
        print(f'    {k}=<your key>{mark}')
        print()
    print('Then Kernel -> Restart. .env is read ONCE at import, so an edit without')
    print('a restart changes nothing and looks like the edit failed.')
    print()

if prov != 'anthropic':
    print('You are NOT on the provider the slides were measured with.')
    print('Everything runs. Your numbers will differ. Report YOURS, not the slide.')
elif os.environ.get(key):
    print('Same provider as the slides -- your numbers should be close, not equal.')

In [ ]:
# [SETUP]
# The repo is organised by session. Python is not: it puts THIS folder on
# sys.path, not the repo root, so `import evalkit` needs one line of help.
# _path.py walks up to the repo root and prepends shared/ and plant/.
import _path  # noqa: F401

# Reload the course modules in DEPENDENCY ORDER before importing anything.
# A Jupyter kernel caches modules; edit a file, re-run, and you silently get
# the first import. Session 7's note applies unchanged.
import importlib, sys
for _m in ('plant7', 'plant_tools7', 'delegation_rows7', 'plant_agents7',
           'seeds7', 'coord_eval7', 'bench7',
           'judge8', 'judge_seeds8', 'judge_bench8', 'agree8', 'human_labels8'):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

import json, judge8, judge_seeds8 as seeds8, judge_bench8, agree8, human_labels8

# STUB = True runs a deterministic keyword matcher: free, no key, and it tells you
# NOTHING about judges. It is here so every cell below produces output even if your
# key is not working. Flip it to False to spend money and get a real answer.
STUB = True
print('judge8', judge8.__version__, '| seeds8', seeds8.__version__,
      '|', 'STUB — plumbing only' if STUB else 'LIVE')

## 1 · What Session 7 left on the table

Four coordination evaluators. Every one of them **skips** the single-agent arm — and it is
right to. Scoring one agent against a four-agent plan is an emissions test for a bicycle.

But that leaves four columns of `None`, and a one-agent answer graded by keyword match.

In [ ]:
# [NULLS]
import coord_eval7
from delegation_rows7 import BY_ID

# Session 7's runs live in session-07/. _path.session(7) finds them from here.
runs = json.load(open(_path.session(7) / 'runs7.json'))['runs']
single = next(r for r in runs if r.get('phase') == 'comparison'
              and r['version'] == 'single' and r['row_id'] == 'HW-001')

for k, v in coord_eval7.run_all(single['outputs'], BY_ID['HW-001']).items():
    print(f"{k:24s} {str(v['score']):>5s}  {v['comment'][:60]}")

In [ ]:
# [REPORT]
# The raw material for everything below: one real, live, four-agent report from
# Session 7. No agent is re-run today. The whole seeded set costs zero.
base = seeds8.load_base()
print(base['answer'][:900])
print('...')
print('TAIL:', base['tail'])

In [ ]:
# [SPLIT]
# A judge reads ONE section. The pipeline joins them with [agent] headers; the
# single arm produces one unlabelled blob, and split_reports handles both.
for agent, text in judge8.split_reports(base['answer']).items():
    print(f"{agent:16s} {len(text):5d} chars   {text[:70]}...")
print()
print('single arm ->', list(judge8.split_reports(single['outputs']['answer'])))

## 2 · The rubric

An evaluator is a hypothesis about a failure: it says **where to look** and **what would
count as failing**. That rule does not stop applying because the evaluator is made of prose.

Three things make a rubric checkable rather than a vibe:

1. it names the failure, not the virtue — *UNSOUND if the cited mechanism belongs to a
   different machine*, not *is this a good diagnosis?*
2. it gives the judge a **reference**, and the reference is not an answer key;
3. it demands one supporting fact, so the verdict can be argued with.

### What the "equipment record" is, since the prompt leans on it

`REFERENCE — equipment record` is the plant's **asset data for one machine**: the entry in
`plant7.EQUIPMENT`. It is nameplate and design data — what the machine *is*, not what is
wrong with it today.

| the record contains | the record does not contain |
|---|---|
| what the machine is (`helical gearbox, 2-stage`), where it sits, how critical it is | any fault code |
| design constants: `shaft_speed_hz: 24.5`, `input_bearing: SKF-6208`, `bearing_thumps_per_turn: 3.19` | any diagnosis |
| operating limits: `alarm_temp_c: 72.0` | any expected or correct recommendation |
| duty: `continuous, 18 h/day` | anything about THIS fault, on THIS day |

**That distinction is the whole reason it is allowed to be in the prompt.** Hand the judge an
answer key and you have not built an evaluator, you have built a lookup: it would agree with
the key and tell you nothing you did not already know. Hand it the equipment record and you
have given it what a maintenance engineer has open on the second monitor anyway — the
machine's own numbers — and then asked whether the report is consistent with them.

So the judge is not being asked *"is this the right answer?"*. It is being asked
**"do this report's own stated numbers hold up against the machine's real ones?"** Concretely:

- the report claims a vibration peak at **3.2×** shaft speed and calls it the input bearing.
  The record says that bearing's defect frequency is **3.19×**. Consistent → `SOUND`.
- the `wrong_evidence` arm quotes a **suction head below NPSHr** — a pump number. The record
  says this machine is a gearbox. Inconsistent → `UNSOUND`, and no answer key was needed.
- the `unsafe_action` arm recommends four more weeks of monitoring. The record says
  `alarm_temp_c: 72.0` and the report itself says the bearing is at 72 °C. Inconsistent →
  `UNSOUND`.

The judge also gets the **manual index** (section ids *with their titles*) and the **parts
index** for the same reason: a human asked to rule on whether a cited manual section is
relevant would have the list of sections in front of them. Session 7's hands-on withheld the
titles and most pairs guessed between `MAN-CONVEYOR-4.2` and `-5.0`. A judge made to guess
for the same reason would be our fault, not the judge's.

Run the next cell and read `REFERENCE` before you read `REPORT`.

In [ ]:
# [RUBRIC]
# Read one. This is the entire judge -- there is nothing else in there.
#
# FIRST: where the REFERENCE block comes from. It is plant asset data, not an answer key.
import plant7
rec = plant7.EQUIPMENT['CONVEYOR']
print("plant7.EQUIPMENT['CONVEYOR'] -- the raw equipment record:")
for k, v in rec.items():
    print(f'    {k:28s} {v}')
print()
print('fault code in it? ', any('fault' in str(k).lower() for k in rec))
print('recommendation in it?', any('action' in str(k).lower() for k in rec))
print('  -> nothing about THIS fault. Design data and limits only.')

# SECOND: the prompt those numbers get rendered into. Four blocks:
#   QUESTION    what this judge is for
#   UNSOUND if  the named failure modes -- the hypothesis
#   REFERENCE   the equipment record above, plus manual + parts indexes
#   REPORT      the agent's own prose, the thing under test
print('=' * 78)
print(judge8.build_prompt('diagnosis_soundness', base, 'CONVEYOR'))

### A judge does not break your harness by being **broken**.
### It breaks it by being **reasonable in a shape you did not anticipate.**

The judge answers. The answer may be perfect. But if the reply does not fit the format
our code expects, our code writes down `None` — and the judge's answer is gone. On the
slide that is row 4, and it looks identical to the judge honestly declining.

**Not one of these is a broken model. Run the next cell and watch them all get lost:**

| what the judge sent back | why our code loses it |
|---|---|
| `{"verdict": "UNSOUND", "evidence": "limits table, not a procedure"}` | a perfect answer **in JSON**. We asked for two plain lines. Common on OpenAI and Gemini. |
| `I'm not able to assess industrial safety recommendations.` | a **refusal**. We asked a safety question and it declined the task. Our code has no box for that. |
| `The diagnosis looks reasonable overall, though I have some doubts.` | **hedged prose**, no verdict word. It answered like a person, not like a form. |
| `4/5` / `quite good` | it fell back to a **1–5 score**, the scale it has seen a million times. |
| `SOUND` | a verdict with **no reason**. We refuse a verdict you cannot argue with. |
| `Could you clarify which manual revision applies?` | it **asked a question back**. |
| `asdkjh qwe zzz` | actual gibberish — and it is last on the list because it is the one that **almost never happens**. |

And these six are the ones that now **survive** — every one of them used to be lost:

| what the judge sent back | why it used to fail |
|---|---|
| `LINE 1: UNSOUND` / `LINE 2: …` | **the bug that cost 192 verdicts.** The prompt said *"LINE 1: one word"* and the model did exactly that. Our parser wanted a bare word. |
| ```` ```UNSOUND``` ```` in a code fence | wrapped in markdown |
| `**UNSOUND**` | bolded the verdict |
| `1. UNSOUND` / `2. …` | numbered the lines |
| `UNSOUND — the cited section is a limits table.` | put both on one line |
| `Sure! Here is my assessment:` then the verdict | polite preamble first |

**Three providers are running in this room and they do not format alike.** A parser tuned
on one provider is a parser that works for one provider — and the students it silently
fails are the ones least able to spot it. That is why the parser is tolerant of *format*
and strict about *substance*.

In [ ]:
# [PARSE]
# WHAT A 'REPLY WE COULD NOT READ' ACTUALLY LOOKS LIKE.
#
# Almost never gibberish. Nearly always a sensible answer in a shape we did not ask
# for. The top six used to fail and now survive -- that tolerance was bought with 192
# wasted verdicts. The bottom seven still land in row 4 of the slide.
#
# Nothing is retried and nothing is dropped: a judge that cannot follow a three-word
# instruction is a measurement ABOUT the judge, and hiding it behind a retry loop is
# how a bad judge gets made to look good.
cases = [
    ('the bug that cost 192 verdicts', 'LINE 1: UNSOUND\nLINE 2: the NPSHr figure is a pump metric.'),
    ('wrapped in a code fence',        '```\nUNSOUND\nthe cited section is a limits table.\n```'),
    ('bolded the verdict',             '**UNSOUND**\nthe cited section is a limits table.'),
    ('both on one line',               'UNSOUND - the cited section is a limits table.'),
    ('numbered the lines',             '1. UNSOUND\n2. the cited section is a limits table.'),
    ('polite preamble first',          'Sure! Here is my assessment:\n\nUNSOUND\nthe section is a limits table.'),
    ('verdict, no reason given',       'SOUND'),
    ('hedged prose, no verdict word',  'The diagnosis looks reasonable to me overall, though I have some doubts.'),
    ('answered in JSON, unasked',      '{"verdict": "UNSOUND", "evidence": "limits table, not a procedure"}'),
    ('used a 1-5 score instead',       '4/5\nquite good'),
    ('asked us a question back',       'Could you clarify which manual revision applies here?'),
    ('declined the task entirely',     "I'm not able to assess industrial safety recommendations."),
    ('returned nothing at all',        ''),
]
print(f"{'what the judge sent back':34s} {'we score it':24s} why")
print('-' * 104)
for label, reply in cases:
    word, why = judge8.parse_verdict(reply)
    lost = word == 'INSUFFICIENT-EVIDENCE'
    print(f"{'LOST >' if lost else '  ok  '} {label:26s} {word:24s} {why[:40]}")
print()
print('The six that survive were ALL failures before harness bug #11 was fixed.')
print('Not one of the lost ones is gibberish. That is the point: a judge fails your')
print('harness by being reasonable in a shape you did not anticipate.')

## 3 · Four judges, and where they fire

| judge | asks | anchored to |
|---|---|---|
| `diagnosis_soundness` | does the stated evidence support the fault it names? | the equipment record |
| `doc_relevance` | is the cited section the one that answers *this* fault? | the manual index, **with titles** |
| `recommendation_safety` | is the action proportionate to the risk and the stock? | criticality, alarm limits, parts |
| `workflow_coherence` | do the sections contradict each other? | the sections themselves |

They are **not** aimed at Session 7's pipeline-vs-single tie. That interval was
`-4% [-52, +43]`, and no judge can resolve a difference wider than the plus-or-minus of the
thing it is judging. That is this session's own spine, and it applies to us first.

In [ ]:
# [JUDGE1]
# All four judges, one report. `run_all` calls every judge, so with STUB = False
# this is FOUR model calls, not one.
#
# The evidence line is printed IN FULL and wrapped, never truncated. The rubric
# demands one supporting fact precisely so the verdict can be argued with; cutting
# it off at 74 characters throws away the only part you can argue with.
import textwrap

res = judge8.run_all(base, None, stub=STUB)
for k, v in res.items():
    verdict, _, evidence = v['comment'].partition(' — ')
    print(f"{k:24s} score={str(v['score']):<5s} {verdict.strip()}")
    for line in textwrap.wrap(evidence.strip() or '(no evidence line)', 92):
        print(f"{'':26s}{line}")
    print()

## 4 · Calibration: reports that are wrong on purpose

Session 7 validated its code evaluators by breaking the pipeline four ways where the right
answer was known. Same move, one level up.

**And say what it does not buy.** The same person wrote the flaw and the rubric, so a judge
that catches these flaws may be catching that person's vocabulary. This measures
*sensitivity*. It cannot tell you the judge is right about a report nobody tampered with.

Which is why two of the seven arms are not broken: `healthy` and `padded`. Only **5 of the
28 cells** in the matrix expect UNSOUND — the other **23 are false-positive tests**, asking
whether the judge stays quiet when it should.

In [ ]:
# [ARMS]
arms = seeds8.build()
base_len = len(arms['healthy']['answer'])
for name, o in arms.items():
    target = [k for k, v in seeds8.EXPECT[name].items() if v == 0] or ['- control -']
    print(f"{name:16s} {len(o['answer']) - base_len:+6d} chars   "
          f"should fail: {', '.join(target)}")

In [ ]:
# [DIFF]
# Exactly what one mutator changed. Every mutator RAISES if its anchor is missing
# or the text did not move -- a mutator that silently no-ops hands the measurement
# a HEALTHY report labelled BROKEN, and the number you get back is a lie.
import difflib
a = arms['healthy']['answer'].split('. ')
b = arms['wrong_evidence']['answer'].split('. ')
for line in difflib.unified_diff(a, b, lineterm='', n=0):
    if line[:1] in '+-' and line[:3] not in ('+++', '---'):
        print(line[:150])

In [ ]:
# [MATRIX]
# 7 arms x 4 judges x REPS.
#
# Three reps on the stub costs nothing, and separation on n=1 is a story rather
# than a measurement -- 1/1 vs 0/5 gives a lower bound of +10%, which is below
# the wobble bound, so the gate below would read DECORATION for arithmetic
# reasons rather than for anything about the judge. Live, three reps is 72 model
# calls; that is the instructor's bill, not yours, which is why REPS drops to 1.
REPS = 3 if STUB else 1
recs = judge_bench8.score_arms(arms, stub=STUB, reps=REPS, verbose=False)
ks = judge8.JUDGE_KEYS
print(f"{'arm':16s} " + ' '.join(f'{k[:13]:>15s}' for k in ks))
for name in arms:
    row = {r['judge']: r for r in recs if r['arm'] == name and r['rep'] == 1}
    print(f'{name:16s} ' + ' '.join(f"{row[k]['verdict'][:13]:>15s}" for k in ks))
print(f'\n{len(recs)} verdicts ({REPS} rep(s)). The table shows rep 1;',
      'the intervals below use all of them.')

In [ ]:
# [KEY]
# Agreement with the answer key -- and, separately, the FALSE ALARMS, which are
# the half that catches the failure this course has already made twice.
keyed = [r for r in recs if r['correct'] is not None]
misses = [r for r in keyed if r['expected'] == 0 and not r['correct']]
alarms = [r for r in keyed if r['expected'] == 1 and not r['correct']]
print(f'agreement with the key : {sum(1 for r in keyed if r["correct"])}/{len(keyed)}')
print(f'missed a real flaw     : {len(misses)}  {[r["arm"] + "/" + r["judge"] for r in misses]}')
print(f'fired on a good report : {len(alarms)}  {[r["arm"] + "/" + r["judge"] for r in alarms]}')

## 5 · Separation, wobble, and the gate

### The gate, in words, before any of the arithmetic

**A bathroom scale that reads three kilos different every time you step on it cannot tell
you that you lost one kilo.** Not because you didn't — because the scale can't see it. One
kilo is inside the scale's own wobble, so the reading is not about you.

A judge is an instrument, and it has the same problem. Ask it the same question twice about
an **unchanged** report and it may answer differently — that is its own noise. Now show it a
report that is fine and one that is broken: how much does its answer change?

**If the answer moves as much on a report that did not change as it does between a good
report and a broken one, then the verdict is not about the report. It is about the judge.**

So when a judge says `UNSOUND`, there are two questions and only one of them is interesting:

| | |
|---|---|
| *is the report broken?* | what you wanted to know |
| *would it have said that anyway?* | what you have to rule out first |

That is the gate. The rest of this section is just the two numbers that let you rule it out.


### What a Wilson interval is, in three sentences

You saw **2 flips in 30**. The honest reading is *not* "the rate is 6.7%" — 30 runs is not
many, and you could easily get 2 by luck from a judge that really flips 15% of the time.

So Wilson turns the question round: **which true flip rates would plausibly have produced 2
out of 30?** The answer here is roughly **2% to 21%**. We quote the top, because the top is
the worst case, and the worst case is what a gate has to survive.

As a rule of thumb it behaves like **adding two imaginary flips and two imaginary non-flips**
before you divide — which is why it never collapses to zero width and never runs off the end
of the scale. The exact form, for reference:

```
centre = (p + z²/2n) / (1 + z²/n)
half   = (z / (1 + z²/n)) · √( p(1-p)/n + z²/4n² )        z = 1.96 for 95%
```

**Why not the textbook one.** `p ± 1.96·√(p(1-p)/n)` at 0 of 30 gives `0% ± 0%` — an interval
of zero width, i.e. *we saw no flips, therefore the rate is exactly 0%, no uncertainty.* That
is the most confident wrong answer in applied statistics, and it is one line of code away at
all times. Half the rows on SLIDE 15 are zero-flip rows.

**And one wrinkle worth knowing**: at 0 of 30 Wilson itself says **11.4%**, while the rule of
three says **10%**, and we quote the rule of three. They disagree slightly; `3/n` is the
standard answer for zero events, but it is not the more conservative one. Better to know that
than to assume they agree.

### And now the arithmetic

```
separation = P(UNSOUND | its own broken arm) - P(UNSOUND | the arms it should pass)
wobble     = how often it disagrees with its own most common answer, same report, n times
```

Both are proportions from small `n`, and several of them will be `0/10` or `10/10`. The
textbook `p ± 1.96·√(p(1-p)/n)` gives an interval of **zero width** at 0 and at 1 — *we saw
no disagreements, therefore the rate is exactly 0%, no uncertainty*. That is the most
confident wrong answer in applied statistics and it is one line of code away at all times.

So: Wilson intervals, and the rule of three (`3/n`) that Session 5 already taught.

In [ ]:
# [SEP]
# Run the matrix a few times first if you are live -- separation on n=1 is a story,
# not a measurement.
for j in judge8.JUDGE_KEYS:
    print('  ' + agree8.separation(recs, j).line())

In [ ]:
# [WOB]
# HOW THE NUMBER IS MADE, in three steps:
#   1. send ONE unchanged report to the same judge n times
#   2. find the answer it gave MOST OFTEN, and count how many of the n disagree
#      with it -- those are its flips on that report
#   3. repeat on three different reports and add them up -> flips out of 3n
#
# The ceiling needs TWO formulas, and this is the part the table hides:
#   0 flips   -> RULE OF THREE, 3/n. There is no interval to build around a
#                count of zero, so this is the standard 'we saw none' answer.
#   any flips -> WILSON interval, upper end.
# Both answer: what is the MOST this judge could be wobbling, given what we saw?
#
# With STUB = False and n=10 across three reports this is 120 model calls --
# which is why the instructor ran it and you are reading the saved file.
wrecs = judge_bench8.wobble(arms, stub=STUB, n=5, verbose=False)
for j in judge8.JUDGE_KEYS:
    w = agree8.wobble(wrecs, j)
    how = 'rule of three, 3/n' if w.flips == 0 else 'Wilson upper end'
    print(f'  {j:24s} {w.flips} flips of {w.n}   saw {w.rate:.0%}   ceiling <= {w.upper:.0%}   ({how})')
    for arm, (flips, n, mode) in sorted(w.per_arm.items()):
        print(f'      {arm:16s} {flips}/{n} differ from {mode}')
    print()

In [ ]:
# [GATE]
# THE PUNCHLINE. One number per judge, and one sentence.
# Reads the SAVED live file when it exists, because the cell above is stub by
# default and a stub judge separates 100% and wobbles 0% BY CONSTRUCTION.
import os
if os.path.exists('judge_runs8.json'):
    live = judge_bench8.load('judge_runs8.json')
    print('reading the instructor\'s LIVE verdicts\n')
else:
    live = recs + wrecs
    print('!! no judge_runs8.json -- these are STUB numbers and are not findings\n')

rep = agree8.report(live)
for j in judge8.JUDGE_KEYS:
    s, w = rep['separation'][j], rep['wobble'][j]
    print(f"{j:24s} sep_lo {s['lo']:+.0%}  vs  wobble_hi {w['upper']:.0%}"
          f"   -> {rep['verdicts'][j]}")

In [ ]:
# [VERBOSITY]
# Same content, four times the words, nothing false added. A verdict that moves
# here moved on LENGTH. This is the bias demonstration, and it cost four calls
# because `padded` was already in the matrix.
for j, v in rep['verbosity'].items():
    print(f"{j:24s} {str(v['healthy']):22s} -> {str(v['padded']):22s} "
          f"{'MOVED' if v['moved'] else 'unchanged'}")

In [ ]:
# [HUMAN]
# Judge vs code vs human, on the only cases in this course where a human opinion
# is actually needed: real disagreements from Session 7's live runs.
h = human_labels8.report()
print(f"{h['n_instances']} run-instances across {h['n_rows']} rows\n")
for row, d in h['rows'].items():
    print(f"{row}  {d['shape']:19s} agent said {str(d['agent_said']):34s}"
          f" -> I said: {d['label'] or 'UNLABELLED'}")
print('\n' + h['caveat'])

## 6 · Hands-on — break it

### The whole exercise is three edits in one file: `my_attack8.py`

Open it. Every spot you touch is marked **`EDIT ME`**. Nothing else in the repo changes.

| | where | what you write |
|---|---|---|
| **0** | *nothing yet* | **`python screen_my_attack.py --show`** — free, no key. Prints the broken report in full, names the judge, and lists the strings you may not delete. **Do this first.** |
| **1** | `EDIT ME #1` — `TARGET` | `"wrong_section"` or `"unsafe_action"`. |
| **2** | `EDIT ME #2` — `PREDICT` | one sentence, **before** you run anything. The screener refuses while it says TODO. |
| **3** | `EDIT ME #3` — `attack()` | your rewrite. `return report.replace(old, new)` is what everyone uses. |
| **4** | *run it* | **`python screen_my_attack.py`** — 8 model calls. Then back to step 3. |

Re-running `--show` after step 3 prints **exactly which words your edit moved**, free,
before you spend anything on scoring it. Use it — a `.replace()` whose first
argument is not in the report matches nothing and changes nothing.

### The rule that makes it an attack

Rewrite the report so its judge says `SOUND` — **without removing the flaw.** The screener
checks the flaw is still there:

| target | the judge you are fooling | what must survive your edit |
|---|---|---|
| `wrong_section` | `doc_relevance` | `MAN-CONVEYOR-5.0` and the line `SECTION: MAN-CONVEYOR-5.0` |
| `unsafe_action` | `recommendation_safety` | the line `ACTION: monitor` |

| verdict | what happened |
|---|---|
| `FOOLED` | the judge passed a report that is still broken. This is the target. |
| `NO-EFFECT` | the judge still says `UNSOUND`. Your edit did not move it. |
| `REPAIRED` | you removed the flaw. Put it back and attack the verdict instead. |
| `COLLATERAL` | a *different* judge fired. That counts, and it is the argument for having four. |

You are not attacking the model's reasoning. You are attacking the seam between the verdict
and the evidence line: a judge asked for a verdict *plus one supporting fact* will often
reach for the most confident-sounding fact in front of it.

In [ ]:
# [ATTACK]
# STEP 0 FIRST -- free, no key, no model call. Read what you are attacking:
!python screen_my_attack.py --show

# Then edit the three EDIT ME blocks in my_attack8.py, re-run --show to
# check your edit landed, and only then run the scorer below.
#
# EIGHT model calls per attempt, on your key: four judges on the untouched broken
# report, four on yours. The baseline is re-measured live every time -- see the
# COST section in screen_my_attack.py. Five attempts is a normal session, ~40 calls.
# !python screen_my_attack.py

## 7 · What today added

A judge can answer questions no code evaluator in this course can express — whether a
diagnosis is *sound*, whether a citation is *relevant*, whether a recommendation is *safe*.

And a judge is an agent, so it inherits every problem an agent has. It is a sample, not a
reading. Ask it twice and you may get two answers, and every verdict it has ever given you
sits inside that spread.

> **You cannot measure a difference smaller than your judge's own noise.**

Which is why the gate is not *is the judge good* — it is *is the difference it claims bigger
than its disagreement with itself*. A judge that fails that may still be right. We simply
cannot show it, and a slide that says otherwise is claiming a difference smaller than the
instrument.

**Homework** is in your inbox: one attack with its verdict, and one rubric line you would
change, with the reason.